# Xem nhanh cấu trúc sample data MBD-mini

Notebook hiển thị **5 dòng đầu của từng bảng** trong `sampledata/mbd_mini/`, kèm tên cột và kiểu dữ liệu Parquet/Arrow.
Sample gồm 1.000 khách được lấy mẫu phân tầng; xem `sampledata/README.md` để biết phương pháp và giới hạn của mẫu.

**Cách chạy:** chọn kernel Python trong `.venv`, sau đó **Run All**.
Nếu thiếu thư viện, chạy trong terminal ở thư mục gốc repo:

```powershell
.\.venv\Scripts\python.exe -m pip install pandas pyarrow ipykernel
```

Mỗi bảng được đọc trực tiếp từ file Parquet sample, lấy theo thứ tự dòng trong file, không sắp xếp theo thời gian.
Cột `fold` chỉ hiển thị nếu có sẵn trong bảng, không suy ra từ thư mục.
Các mảng dài được rút gọn khi hiển thị; DataFrame mẫu vẫn giữ nguyên giá trị.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import Markdown, display

# Tìm repo khi chạy từ thư mục gốc, notebooks/ hoặc thư mục con.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "sampledata" / "mbd_mini").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Không tìm thấy sampledata/mbd_mini/. Hãy chạy notebook trong repo này.")

DATA_DIR = PROJECT_ROOT / "sampledata" / "mbd_mini"
N_ROWS = 5
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_seq_items", 6)
np.set_printoptions(threshold=6, edgeitems=2)
print(f"Thư mục dữ liệu: {DATA_DIR}")


## Các bảng dữ liệu

`dialog`, `geo`, `trx` lưu từng sự kiện; `ptls_*` gom chuỗi sự kiện của khách hàng thành các mảng.
`clients` chứa khách hàng và fold, còn `targets` chứa nhãn theo tháng.
Các bảng phụ mô tả việc lấy mẫu và lưới khách hàng theo tháng.


In [ ]:
TABLES = {
    "clients": "Khách hàng được chọn và fold",
    "targets": "Nhãn sản phẩm và thông tin theo tháng",
    "dialog": "Từng lượt hội thoại và embedding",
    "geo": "Từng sự kiện vị trí",
    "trx": "Từng giao dịch",
    "ptls_dialog": "Chuỗi hội thoại của khách hàng",
    "ptls_geo": "Chuỗi sự kiện vị trí của khách hàng",
    "ptls_trx": "Chuỗi giao dịch của khách hàng",
    "selection": "Tầng lấy mẫu và trọng số của khách hàng",
    "customer_month": "Lưới khách hàng theo tháng và số log",
    "coverage_selection": "Khách minh họa theo nhóm có chủ đích",
}
parquet_files = {name: DATA_DIR / f"{name}.parquet" for name in TABLES}
missing = [str(path) for path in parquet_files.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Không tìm thấy file Parquet: {missing}")

display(pd.DataFrame([
    {"Bảng": name, "Mô tả": description, "File Parquet": parquet_files[name].name}
    for name, description in TABLES.items()
]))


## Hàm xem mẫu

Đọc batch đầu tiên từ mỗi file Parquet sample, tối đa 5 dòng.
Bảng schema hiển thị kiểu dữ liệu gốc, bao gồm thời gian và mảng lồng nhau.


In [ ]:
def preview_table(name):
    path = parquet_files[name]
    with pq.ParquetFile(path) as parquet:
        batch = next(parquet.iter_batches(batch_size=N_ROWS), None)
        sample = (
            pa.Table.from_batches([batch])
            if batch is not None
            else pa.Table.from_batches([], schema=parquet.schema_arrow)
        )
    frame = sample.to_pandas()

    display(Markdown(f"**{len(frame)} dòng mẫu · {len(sample.schema)} cột**"))
    display(frame)
    display(Markdown("**Tên cột và kiểu dữ liệu**"))
    display(pd.DataFrame([
        {"Cột": field.name, "Kiểu Arrow": str(field.type), "Kiểu pandas": str(frame[field.name].dtype)}
        for field in sample.schema
    ]))
    return frame


## clients

Khách hàng và fold dùng cho cross-validation.


In [ ]:
clients = preview_table("clients")


## targets

Nhãn sản phẩm và thông tin khách hàng theo tháng.


In [ ]:
targets = preview_table("targets")


## dialog

Mỗi dòng là một sự kiện hội thoại, có embedding dạng mảng.


In [ ]:
detail_dialog = preview_table("dialog")


## geo

Mỗi dòng là một sự kiện vị trí.


In [ ]:
detail_geo = preview_table("geo")


## trx

Mỗi dòng là một giao dịch.


In [ ]:
detail_trx = preview_table("trx")


## ptls_dialog

Dữ liệu hội thoại dạng chuỗi; một số cột là mảng hoặc mảng lồng nhau.


In [ ]:
ptls_dialog = preview_table("ptls_dialog")


## ptls_geo

Dữ liệu vị trí dạng chuỗi.


In [ ]:
ptls_geo = preview_table("ptls_geo")


## ptls_trx

Dữ liệu giao dịch dạng chuỗi.


In [ ]:
ptls_trx = preview_table("ptls_trx")


## selection


In [ ]:
selection = preview_table("selection")


## customer_month


In [ ]:
customer_month = preview_table("customer_month")


## coverage_selection


In [ ]:
coverage_selection = preview_table("coverage_selection")
